# Anthropic Messages API: Making Your First Call

This notebook provides a complete, instructional guide to using the official **Anthropic Python SDK** (`anthropic`) to query Claude models.

---

### Core Concepts to Know
1. **The Messages API**: Claude uses a structured conversation format with `messages` (`role: "user"` and `role: "assistant"`).
2. **Mandatory `max_tokens`**: In the Anthropic SDK, `max_tokens` is **required** on every request.
3. **System Prompts**: System instructions are passed via a top-level `system` parameter, **not** inside the `messages` array.
4. **Exact Model Identifiers**: Anthropic model strings follow specific naming conventions (e.g. `claude-3-5-haiku-latest`, `claude-3-5-sonnet-latest`, `claude-3-7-sonnet-latest`).

> ⚠️ **Note:** To conserve your API credits, do not run the request cells until you are ready to test.

## Step 1: Install & Import Required Libraries

If you haven't installed the dependencies yet, run from your project root:
```bash
pip install -r requirements.txt
```

Now import the required modules:

In [ ]:
import os
import anthropic
from dotenv import load_dotenv, find_dotenv

## Step 2: Securely Load Your API Key

Never hardcode API keys into your code or notebooks. Store your credentials in a `.env` file in your project root:
```env
ANTHROPIC_API_KEY=your-api-key-here
# or:
API_KEY=your-api-key-here
```

The code below uses `find_dotenv()` to locate the root `.env` file and validates that an API key is present:

In [ ]:
# Automatically finds and loads .env from the project root
load_dotenv(find_dotenv())

# Check for standard ANTHROPIC_API_KEY or custom API_KEY
api_key = os.getenv("ANTHROPIC_API_KEY") or os.getenv("API_KEY")

if not api_key:
    raise ValueError(
        "API Key not found! Please check that your .env file defines ANTHROPIC_API_KEY or API_KEY."
    )

print("✅ API Key successfully detected in environment.")

## Step 3: Initialize the Anthropic Client

Create an instance of `anthropic.Anthropic`. 

*Tip:* If your environment variable is named `ANTHROPIC_API_KEY`, the client automatically discovers it without needing `api_key=...` explicitly passed.

In [ ]:
client = anthropic.Anthropic(api_key=api_key)

## Step 4: Construct and Send the API Request

Key parameters for `client.messages.create()`:
- **`model`** *(required)*: The target Claude model. Examples:
  - `"claude-3-5-haiku-latest"` (fastest & most cost-effective)
  - `"claude-3-5-sonnet-latest"` (balanced intelligence & speed)
  - `"claude-3-7-sonnet-latest"` (hybrid reasoning & frontier capability)
- **`max_tokens`** *(required)*: The maximum tokens Claude is allowed to generate.
- **`messages`** *(required)*: List of message objects representing dialogue turns.
- **`system`** *(optional)*: System instructions setting Claude's behavior or persona.

In [ ]:
# Note: Running this cell sends a live request to the Anthropic API and consumes tokens.
message = client.messages.create(
    model="claude-3-5-haiku-latest",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Hello, world!"}
    ],
)

## Step 5: Parse and Inspect the Response

The API returns a `Message` response object with several useful attributes:
- **`message.content`**: A list of blocks (such as `TextBlock(type='text', text='...')`).
- **`message.stop_reason`**: Why generation stopped (e.g., `"end_turn"` or `"max_tokens"`).
- **`message.usage`**: Token metrics (`input_tokens` and `output_tokens`).

In [ ]:
# 1. Display Claude's text content
for block in message.content:
    if block.type == "text":
        print(block.text)

# 2. Inspect metadata
print(f"\nStop Reason: {message.stop_reason}")
print(f"Token Usage: {message.usage.input_tokens} input, {message.usage.output_tokens} output")

## Step 6: Recommended Pattern (Production Error Handling)

In production applications, wrap your calls in a `try...except` block to catch specific Anthropic API exceptions without crashing your workflow:

In [ ]:
try:
    response = client.messages.create(
        model="claude-3-5-haiku-latest",
        max_tokens=500,
        messages=[{"role": "user", "content": "Explain latency in one sentence."}],
    )
    print(response.content[0].text)
except anthropic.APIConnectionError as e:
    print(f"Network error connecting to Anthropic: {e}")
except anthropic.RateLimitError as e:
    print(f"Rate limit exceeded: {e}")
except anthropic.APIStatusError as e:
    print(f"API error (Status {e.status_code}): {e.response}")